# Lời giải bài tập thuật toán MapReduce

## Bài 1 — Tần suất hai sản phẩm được mua cùng nhau

**Đầu vào:** `transactions` là danh sách các giỏ hàng; mỗi giỏ chứa các mã sản phẩm có thể sắp xếp theo cùng một quy tắc.

**Đầu ra:** từ điển có khóa là cặp sản phẩm `(a, b)` với `a < b`, giá trị là số giao dịch chứa cả hai sản phẩm. Điều kiện `a < b` chỉ quy định thứ tự mã sản phẩm để mỗi cặp có một khóa thống nhất, không so sánh giá hoặc số lượng sản phẩm.

Một sản phẩm xuất hiện nhiều lần trong cùng giỏ chỉ được tính một lần. Các giỏ ở những dòng khác nhau được tính là các giao dịch riêng biệt; nếu có dòng trùng do lỗi nhập, cần khử trùng theo mã giao dịch trước khi đưa danh sách giỏ hàng vào hàm.

### Cách thực hiện

1. **Map:** dùng `set` loại bỏ sản phẩm trùng trong mỗi giỏ, rồi dùng `sorted` sắp xếp các mã sản phẩm. Hàm `combinations(..., 2)` tạo các cặp gồm hai sản phẩm khác nhau. Mỗi cặp đóng góp một bản ghi có khóa là cặp sản phẩm và giá trị bằng `1`.
2. **Shuffle:** gom các giá trị có cùng khóa vào một danh sách trong `shuffled`. Nhờ sắp xếp trước, các giao dịch chứa cùng cặp sản phẩm luôn dùng cùng một khóa, dù thứ tự sản phẩm trong giỏ khác nhau.
3. **Reduce:** cộng các giá trị của từng khóa để thu được số giao dịch chứa cặp sản phẩm đó. Giỏ rỗng hoặc chỉ có một sản phẩm khác nhau không tạo ra cặp nào.
4. **Combiner (tùy chọn):** cộng gộp số đếm của các cặp sản phẩm cùng khóa tại mỗi mapper, giúp giảm lượng dữ liệu truyền giữa các máy. Reducer cộng các số đếm cục bộ để thu được tổng số giao dịch chứa từng cặp. Combiner có thể không được chạy hoặc được chạy nhiều lần; phép cộng vẫn cho cùng kết quả.

Combiner được liệt kê ở mục 4 để trình bày phần tùy chọn; khi sử dụng, thứ tự xử lý là **Map → Combiner → Shuffle → Reduce**. Mã Python dưới đây chưa triển khai Combiner.

### Mã Python

In [1]:
from collections import defaultdict
from itertools import combinations


def cooccurrence(transactions):
    """Mỗi giỏ đóng góp tối đa 1 lần cho một cặp sản phẩm không thứ tự."""
    shuffled = defaultdict(list)
    for basket in transactions:
        for pair in combinations(sorted(set(basket)), 2):  # Map
            shuffled[pair].append(1)  # Shuffle
    return {pair: sum(values) for pair, values in shuffled.items()}

### Ví dụ kiểm chứng

Dùng các giỏ hàng bên dưới để kiểm tra cặp sản phẩm đảo thứ tự, sản phẩm lặp trong cùng giỏ, giỏ rỗng và giỏ chỉ có một sản phẩm. Đây là dữ liệu minh họa được tạo để kiểm tra thuật toán.

Đếm trực tiếp: cặp `A, B` có trong giỏ 1, 2, 3 nên bằng **3**; cặp `A, C` có trong giỏ 3, 4 nên bằng **2**; cặp `B, C` chỉ có trong giỏ 3 nên bằng **1**.

In [2]:
transactions = [
    ['A', 'B'],
    ['B', 'A', 'A'],
    ['A', 'B', 'C'],
    ['A', 'C'],
    [],
    ['C'],
]
expected_counts = {('A', 'B'): 3, ('A', 'C'): 2, ('B', 'C'): 1}
actual_counts = cooccurrence(transactions)

for index, basket in enumerate(transactions, start=1):
    print(f'Giỏ {index}: {basket}')
print('\nCặp sản phẩm | Kết quả | Mong đợi')
for pair in sorted(expected_counts):
    print(f'{pair} | {actual_counts.get(pair)} | {expected_counts[pair]}')

assert actual_counts == expected_counts, 'Số đếm chưa khớp với kết quả tính trực tiếp.'
assert cooccurrence([]) == {}
print('\nĐẠT: mọi số đếm khớp với kết quả mong đợi; đầu vào rỗng trả về {}.')

Giỏ 1: ['A', 'B']
Giỏ 2: ['B', 'A', 'A']
Giỏ 3: ['A', 'B', 'C']
Giỏ 4: ['A', 'C']
Giỏ 5: []
Giỏ 6: ['C']

Cặp sản phẩm | Kết quả | Mong đợi
('A', 'B') | 3 | 3
('A', 'C') | 2 | 2
('B', 'C') | 1 | 1

ĐẠT: mọi số đếm khớp với kết quả mong đợi; đầu vào rỗng trả về {}.


## Bài 2 — K-Means dạng lặp

**Đầu vào:** `points` gồm `N` điểm có cùng số chiều `d`; `initial_centers` gồm `K` tâm ban đầu; `tol` là sai số hữu hạn không âm; `max_iter` là số nguyên dương giới hạn số vòng lặp. Dữ liệu và bộ tâm phải không rỗng, số chiều phải dương và mọi tọa độ phải hữu hạn. Trong bài toán phân cụm, chọn `1 ≤ K ≤ N`.

**Đầu ra:** `centers` là bộ tâm cuối cùng, `history` lưu số thứ tự vòng lặp, bộ tâm sau cập nhật và độ dịch chuyển tâm lớn nhất của từng vòng.

### Cách thực hiện

Mỗi lần gọi `kmeans_step` tương ứng với một vòng cập nhật theo MapReduce:

1. **Map:** tính bình phương khoảng cách Euclid từ mỗi điểm đến các tâm và chọn tâm gần nhất. Nếu khoảng cách bằng nhau, chọn cụm có chỉ số nhỏ nhất. Bản ghi đầu ra có khóa là chỉ số cụm và giá trị là `(điểm, 1)`.
2. **Shuffle:** gom các bản ghi có cùng chỉ số cụm vào `shuffled`.
3. **Reduce:** cộng tọa độ theo từng chiều rồi chia cho tổng số điểm trong cụm để tính tâm mới. Cụm không có điểm giữ nguyên tâm cũ, tránh phép chia cho 0.
4. **Combiner (tùy chọn):** gộp các bản ghi cùng chỉ số cụm thành `(tổng vector, số điểm)` tại mỗi mapper. Reducer cộng các tổng vector và số điểm rồi mới chia; không lấy trung bình trực tiếp các trung bình vì số điểm của các nhóm có thể khác nhau.

Khi sử dụng Combiner, thứ tự xử lý là **Map → Combiner → Shuffle → Reduce**. Mã Python dưới đây chưa triển khai Combiner.

Hàm `kmeans` đóng vai trò **Driver**: gọi các vòng cập nhật, tính độ dịch chuyển Euclid lớn nhất giữa tâm mới và tâm cũ, rồi dừng khi độ dịch chuyển không quá `tol` hoặc đã chạy đủ `max_iter` vòng. Các tâm được đối chiếu theo chỉ số cụm. Hàm `_validate` kiểm tra dữ liệu trước khi tính toán.

### Mã Python

In [3]:
from collections import defaultdict
from math import isfinite, sqrt


def _validate(points, centers):
    if not points or not centers or not centers[0]:
        raise ValueError('Cần dữ liệu và ít nhất một tâm có số chiều dương.')
    dim = len(centers[0])
    if any(len(x) != dim or not all(isfinite(v) for v in x) for x in points + centers):
        raise ValueError('Số chiều phải đồng nhất; tọa độ phải hữu hạn.')


def kmeans_step(points, centers):
    """Một job MapReduce. Cụm rỗng giữ tâm cũ; hòa khoảng cách chọn ID nhỏ."""
    points = [tuple(x) for x in points]
    centers = [tuple(c) for c in centers]
    _validate(points, centers)
    shuffled = defaultdict(list)
    for x in points:
        cluster = min(range(len(centers)),
                      key=lambda j: sum((a-b)**2 for a,b in zip(x,centers[j])))
        shuffled[cluster].append((x, 1))  # Map: (cluster_id, (vector, count))
    updated = centers.copy()
    for cluster, values in shuffled.items():
        count = sum(n for _,n in values)
        updated[cluster] = tuple(sum(x[d] for x,_ in values)/count
                                 for d in range(len(centers[0])))
    return updated


def kmeans(points, initial_centers, tol=1e-4, max_iter=100):
    if tol < 0 or not isfinite(tol) or max_iter < 1:
        raise ValueError('tol phải hữu hạn >= 0; max_iter >= 1.')
    centers = [tuple(c) for c in initial_centers]
    points = [tuple(x) for x in points]
    _validate(points, centers)
    history = []
    for iteration in range(1, max_iter + 1):
        new_centers = kmeans_step(points, centers)
        shift = max(sqrt(sum((a-b)**2 for a,b in zip(old,new)))
                    for old,new in zip(centers,new_centers))
        history.append({'iteration': iteration, 'centers': new_centers, 'max_shift': shift})
        centers = new_centers
        if shift <= tol:
            break
    return centers, history

### Điều kiện dừng

Dừng theo `tol` là tiêu chuẩn gần đúng. Dừng vì đạt `max_iter` không đồng nghĩa đã hội tụ; có thể xem `max_shift` ở phần tử cuối của `history` để biết độ dịch chuyển cuối cùng. Hàm trả về các tâm và lịch sử cập nhật; nếu cần nhãn của từng điểm, phải gán lại điểm theo bộ tâm cuối.

### Ví dụ kiểm chứng

Dùng bốn điểm `(0, 0)`, `(2, 2)`, `(10, 10)`, `(12, 12)` với hai tâm ban đầu `(0, 0)` và `(10, 10)`.

Sau một vòng cập nhật, hai điểm đầu thuộc cụm 0, hai điểm sau thuộc cụm 1. Trung bình tọa độ cho hai tâm mới là **`(1, 1)` và `(11, 11)`**. Vòng tiếp theo giữ nguyên các tâm, nên độ dịch chuyển bằng `0` và Driver dừng. Đoạn mã đối chiếu cả kết quả một vòng và kết quả chạy đến điều kiện dừng.

In [4]:
points = [(0., 0.), (2., 2.), (10., 10.), (12., 12.)]
initial_centers = [(0., 0.), (10., 10.)]
expected_centers = [(1., 1.), (11., 11.)]
tol = 1e-4

one_step = kmeans_step(points, initial_centers)
assert one_step == expected_centers
print('Tâm sau một vòng:', one_step)

centers, history = kmeans(points, initial_centers, tol=tol, max_iter=100)
print('\nVòng | Các tâm sau cập nhật | Dịch chuyển lớn nhất')
for row in history:
    print(f"{row['iteration']} | {row['centers']} | {row['max_shift']:.6f}")

assert centers == expected_centers
assert len(history) == 2
assert history[-1]['max_shift'] == 0
assert history[-1]['max_shift'] <= tol
print('\nĐẠT: tâm cuối khớp với kết quả tính trực tiếp; Driver dừng sau 2 vòng.')

Tâm sau một vòng: [(1.0, 1.0), (11.0, 11.0)]

Vòng | Các tâm sau cập nhật | Dịch chuyển lớn nhất
1 | [(1.0, 1.0), (11.0, 11.0)] | 1.414214
2 | [(1.0, 1.0), (11.0, 11.0)] | 0.000000

ĐẠT: tâm cuối khớp với kết quả tính trực tiếp; Driver dừng sau 2 vòng.


### Kiểm tra các trường hợp đặc biệt

Kiểm tra thêm chính sách giữ tâm của cụm rỗng, chọn chỉ số cụm nhỏ nhất khi khoảng cách bằng nhau và dừng khi đã chạy đủ `max_iter` vòng.

In [5]:
empty_cluster_centers = kmeans_step(
    points, [(0., 0.), (10., 10.), (99., 99.)]
)
assert empty_cluster_centers == [(1., 1.), (11., 11.), (99., 99.)]
print('Cụm rỗng: tâm cuối vẫn là', empty_cluster_centers[2])

tie_centers = kmeans_step([(1.,)], [(0.,), (2.,)])
assert tie_centers == [(1.,), (2.,)]
print('Khoảng cách bằng nhau: điểm được gán cho cụm 0; các tâm mới là', tie_centers)

limited_centers, limited_history = kmeans(points, initial_centers, max_iter=1)
assert len(limited_history) == 1
assert limited_history[-1]['max_shift'] > tol
print('Giới hạn max_iter=1: dừng sau', len(limited_history), 'vòng dù độ dịch chuyển còn lớn hơn tol.')
print('\nĐẠT: cả 3 trường hợp đặc biệt cho kết quả mong đợi.')

Cụm rỗng: tâm cuối vẫn là (99.0, 99.0)
Khoảng cách bằng nhau: điểm được gán cho cụm 0; các tâm mới là [(1.0,), (2.0,)]
Giới hạn max_iter=1: dừng sau 1 vòng dù độ dịch chuyển còn lớn hơn tol.

ĐẠT: cả 3 trường hợp đặc biệt cho kết quả mong đợi.
